In [1]:
from pyspark.sql import functions as F
 
 
SILVER_SCHEMA = "silver"
OBT_TABLE_NAME = "sales_obt"

StatementMeta(, d9a30a51-ec5d-4c2f-9ae4-1061b57ce105, 3, Finished, Available, Finished, False)

In [2]:
OBT_METADATA = [
    {
        "table": "orders_t",
        "alias": "orders",
        "column_alias_prefix": "order",
        "is_base": True,
        "join_type": None,
        "join_condition": None,
        "columns": [
            "order_id", "customer_id", "store_id", "order_timestamp",
            "payment_method", "order_status", "total_amount",
        ],
        "audit_columns": ["created_timestamp", "updated_timestamp", "processed_at", "is_active"],
    },
    {
        "table": "customers_t",
        "alias": "customers",
        "column_alias_prefix": "customer",
        "is_base": False,
        "join_type": "LEFT JOIN",
        "join_condition": "orders.customer_id = customers.customer_id",
        "columns": [
            "first_name", "last_name", "email", "phone",
            "city", "province", "country",
        ],
        "audit_columns": ["created_timestamp", "updated_timestamp", "processed_at", "is_active"],
    },
    {
        "table": "order_items_t",
        "alias": "order_items",
        "column_alias_prefix": "order_item",
        "is_base": False,
        "join_type": "LEFT JOIN",
        "join_condition": "orders.order_id = order_items.order_id",
        "columns": [
            "order_item_id", "product_id", "quantity",
            "unit_price", "line_amount",
        ],
        "audit_columns": ["created_timestamp", "updated_timestamp", "processed_at", "is_active"],
    },
    {
        "table": "products_t",
        "alias": "products",
        "column_alias_prefix": "product",
        "is_base": False,
        "join_type": "LEFT JOIN",
        "join_condition": "order_items.product_id = products.product_id",
        "columns": [
            "product_name", "category", "brand", "price",
        ],
        "audit_columns": ["created_timestamp", "updated_timestamp", "processed_at", "is_active"],
    },
    {
        "table": "stores_t",
        "alias": "stores",
        "column_alias_prefix": "store",
        "is_base": False,
        "join_type": "LEFT JOIN",
        "join_condition": "orders.store_id = stores.store_id",
        "columns": [
            "store_name", "city", "province", "country",
        ],
        "audit_columns": ["created_timestamp", "updated_timestamp", "processed_at", "is_active"],
    },
    {
        "table": "employees_t",
        "alias": "employees",
        "column_alias_prefix": "employee",
        "is_base": False,
        "join_type": "LEFT JOIN",
        "join_condition": "orders.store_id = employees.store_id",
        "columns": [
            "employee_id", "first_name", "last_name",
            "email", "job_title", "salary",
        ],
        "audit_columns": ["created_timestamp", "updated_timestamp", "processed_at", "is_active"],
    },
]
OBT_EXTRA_COLUMNS = [
    {"expression": "current_timestamp()", "alias": "obt_processed_at"},
]
 

StatementMeta(, d9a30a51-ec5d-4c2f-9ae4-1061b57ce105, 4, Finished, Available, Finished, False)

In [3]:
def get_base_table(metadata: list) -> dict:
    """Return the single table entry flagged as the FROM (base) table."""
    base_tables = [t for t in metadata if t.get("is_base")]
    if len(base_tables) != 1:
        raise ValueError(
            f"Expected exactly one base table (is_base=True), found {len(base_tables)}."
        )
    return base_tables[0]
 
 
def get_join_tables(metadata: list) -> list:
    """Return all non-base tables, in the order they should be joined."""
    return [t for t in metadata if not t.get("is_base")]

StatementMeta(, d9a30a51-ec5d-4c2f-9ae4-1061b57ce105, 5, Finished, Available, Finished, False)

In [4]:
AMBIGUOUS_COLUMNS = {"first_name", "last_name", "email", "city", "province", "country", "phone"}
 
 
def build_select_clause(metadata: list, extra_columns: list = None) -> str:
    extra_columns = extra_columns if extra_columns is not None else []
    select_items = []
 
    for table_meta in metadata:
        alias = table_meta["alias"]
        prefix = table_meta.get("column_alias_prefix", alias)
 
        for column in table_meta.get("columns", []):
            output_name = f"{prefix}_{column}" if column in AMBIGUOUS_COLUMNS else column
            select_items.append(f"{alias}.{column} AS {output_name}")
 
        for column in table_meta.get("audit_columns", []):
            select_items.append(f"{alias}.{column} AS {prefix}_{column}")
 
    for extra in extra_columns:
        select_items.append(f"{extra['expression']} AS {extra['alias']}")
 
    return "SELECT\n    " + ",\n    ".join(select_items)
 
 
def build_from_clause(base_table: dict, schema: str = SILVER_SCHEMA) -> str:
    return f"FROM {schema}.{base_table['table']} AS {base_table['alias']}"
 
 
def build_join_clause(join_tables: list, schema: str = SILVER_SCHEMA) -> str:
    join_lines = []
    for table_meta in join_tables:
        join_type = table_meta.get("join_type", "LEFT JOIN")
        table = table_meta["table"]
        alias = table_meta["alias"]
        condition = table_meta["join_condition"]
 
        if not condition:
            raise ValueError(f"Table '{table}' is missing a join_condition.")
 
        join_lines.append(
            f"{join_type} {schema}.{table} AS {alias}\n    ON {condition}"
        )
 
    return "\n".join(join_lines)
 
 
def build_dynamic_sql(metadata: list, schema: str = SILVER_SCHEMA,
                       extra_columns: list = None) -> str:
    extra_columns = extra_columns if extra_columns is not None else OBT_EXTRA_COLUMNS
    base_table = get_base_table(metadata)
    join_tables = get_join_tables(metadata)
 
    select_clause = build_select_clause(metadata, extra_columns)
    from_clause = build_from_clause(base_table, schema)
    join_clause = build_join_clause(join_tables, schema)
 
    dynamic_sql = f"{select_clause}\n{from_clause}\n{join_clause}"
    return dynamic_sql

StatementMeta(, d9a30a51-ec5d-4c2f-9ae4-1061b57ce105, 6, Finished, Available, Finished, False)

In [5]:
dynamic_sql = build_dynamic_sql(OBT_METADATA, schema=SILVER_SCHEMA)
 
print("=" * 80)
print("GENERATED SQL")
print("=" * 80)
print(dynamic_sql)
print("=" * 80)
 
df = spark.sql(dynamic_sql)
 
# Quick sanity check before writing — row count and schema
print(f"[INFO] OBT row count: {df.count()}")
df.printSchema()

StatementMeta(, d9a30a51-ec5d-4c2f-9ae4-1061b57ce105, 7, Finished, Available, Finished, False)

GENERATED SQL
SELECT
    orders.order_id AS order_id,
    orders.customer_id AS customer_id,
    orders.store_id AS store_id,
    orders.order_timestamp AS order_timestamp,
    orders.payment_method AS payment_method,
    orders.order_status AS order_status,
    orders.total_amount AS total_amount,
    orders.created_timestamp AS order_created_timestamp,
    orders.updated_timestamp AS order_updated_timestamp,
    orders.processed_at AS order_processed_at,
    orders.is_active AS order_is_active,
    customers.first_name AS customer_first_name,
    customers.last_name AS customer_last_name,
    customers.email AS customer_email,
    customers.phone AS customer_phone,
    customers.city AS customer_city,
    customers.province AS customer_province,
    customers.country AS customer_country,
    customers.created_timestamp AS customer_created_timestamp,
    customers.updated_timestamp AS customer_updated_timestamp,
    customers.processed_at AS customer_processed_at,
    customers.is_act

In [6]:
full_obt_name = f"{SILVER_SCHEMA}.{OBT_TABLE_NAME}"
print(f"[WRITE] {full_obt_name}")
 
(
    df.write
      .format("delta")
      .mode("overwrite")
      .option("overwriteSchema", "true")
      .saveAsTable(full_obt_name)
)
 
print(f"OBT build complete: {full_obt_name}")

StatementMeta(, d9a30a51-ec5d-4c2f-9ae4-1061b57ce105, 8, Finished, Available, Finished, False)

[WRITE] silver.sales_obt
OBT build complete: silver.sales_obt


In [7]:
%%sql
select * from silver.sales_obt limit 2;

StatementMeta(, d9a30a51-ec5d-4c2f-9ae4-1061b57ce105, 9, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 58 fields>

In [8]:
%%sql
select * from silver.orders_t limit 2;

StatementMeta(, d9a30a51-ec5d-4c2f-9ae4-1061b57ce105, 10, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 11 fields>